In [1]:
# Run first
!pip install -q transformers datasets accelerate evaluate gradio langgraph

# Show versions (optional)
import transformers, datasets, gradio
print("transformers", transformers.__version__, "datasets", datasets.__version__, "gradio", gradio.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
transformers 5.0.0 datasets 4.0.0 gradio 5.50.0


In [24]:
import io, pandas as pd, os
from datasets import Dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import torch
from transformers import pipeline
import random
import re
from typing import TypedDict, Optional, Dict, Any

# LangGraph imports
from langgraph.graph import StateGraph, END

# Gradio
import gradio as gr

### Dataset to train the model

In [5]:
### Training dataset
csv_path = r'/content/banking_support_dataset_train_model.csv'
df = pd.read_csv(csv_path)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
# print(df.head())

# Quick label sanity check
expected = {"Query","Positive Feedback","Negative Feedback"}
labels = set(df['label'].astype(str).unique().tolist())
print("Unique labels in file:", labels)
if not labels.issubset(expected):
    raise SystemExit(f"CSV contains unexpected labels. Allowed labels: {expected}")

df.head()

Rows: 300
Columns: ['text', 'label']
Unique labels in file: {'Query', 'Positive Feedback', 'Negative Feedback'}


,text,label
0,Excellent service on my credit card query.,Positive Feedback
1,I’m happy with the way you resolved my cheque ...,Positive Feedback
2,Your support team was very helpful with my FD ...,Positive Feedback
3,Your support team was very helpful with my FD ...,Positive Feedback
4,I’m happy with the way you resolved my cheque ...,Positive Feedback


## Fine tune a transformer model

In [6]:
# Fine-tune a small transformer classifier on your CSV

## Reference: https://huggingface.co/distilbert/distilbert-base-uncased

# Config
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "/content/intent-distilbert"
NUM_LABELS = 3
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

# Load dataset from pandas DataFrame 'df' created earlier
label_list = ["Query","Positive Feedback","Negative Feedback"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

# Map labels to ids
df['label_id'] = df['label'].map(label2id)

# Create Hugging Face Dataset
dataset = Dataset.from_pandas(df[['text','label_id']].rename(columns={'text':'text','label_id':'label'}))
# Shuffle and split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = dataset['train']
eval_ds = dataset['test']

# Tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def preprocess(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)
train_ds = train_ds.map(preprocess, batched=True)
eval_ds = eval_ds.map(preprocess, batched=True)
train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
eval_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id)

# Metrics
import evaluate
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    do_eval=True,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=10,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save tokenizer and model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved to", OUTPUT_DIR)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.027656
20,0.795320
30,0.539320
40,0.385246
50,0.313558


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/intent-distilbert


### Model Classifier function

In [8]:
# After training, load the fine-tuned model and use HF pipeline for inference
from transformers import pipeline

MODEL_DIR = "/content/intent-distilbert"
classifier = pipeline("text-classification", model=MODEL_DIR, tokenizer=MODEL_DIR, return_all_scores=False)

def hf_model_classify(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "":
        return "Query"
    out = classifier(text)[0]
    # out example: {'label': 'Positive Feedback', 'score': 0.98}
    label = out['label']
    # Some pipelines return labels like "LABEL_0" depending on save; normalize if needed
    if label not in ["Query","Positive Feedback","Negative Feedback"]:
        # try mapping by id if label looks like LABEL_X
        if label.startswith("LABEL_"):
            idx = int(label.split("_")[1])
            mapping = {0:"Query",1:"Positive Feedback",2:"Negative Feedback"}
            return mapping.get(idx, "Query")
        return "Query"
    return label


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [9]:
# Sample Test Cases
samples = [
    "Thanks, that was helpful",
    "My card was declined and I need help",
    "What is the status of 123456"
]
for s in samples:
    print(s, "->", hf_model_classify(s))

Thanks, that was helpful -> Positive Feedback
My card was declined and I need help -> Negative Feedback
What is the status of 123456 -> Query


In [13]:
# -------------------------
# Config / Paths
# -------------------------
CSV_FILE = "/content/support_tickets.csv"

# Ensure CSV exists
if not os.path.exists(CSV_FILE):
    df = pd.DataFrame(columns=["ticket_id", "status", "message", "customer_name", "satisfaction"])
    df.to_csv(CSV_FILE, index=False)
    print(f"Created empty CSV file: {CSV_FILE}")

Created empty CSV file: /content/support_tickets.csv


### Generate unique ticket

In [15]:
def generate_unique_ticket_id():
    df = pd.read_csv(CSV_FILE)
    existing_ids = set(df['ticket_id'].dropna().astype(int).tolist()) if not df.empty else set()
    while True:
        ticket_id = random.randint(100000, 999999)
        if ticket_id not in existing_ids:
            return ticket_id

### Feedback handler function

In [16]:
def handle_feedback(user_message, customer_name):
    # classification = ml_classify_message(user_message)
    classification = hf_model_classify(user_message)
    if classification == "Positive Feedback":
        return classification, f"Thank you for your kind words, {customer_name}! We’re delighted to assist you.", None
    elif classification == "Negative Feedback":
        ticket_id = generate_unique_ticket_id()
        new_ticket = {
            "ticket_id": ticket_id,
            "status": "Unresolved",
            "message": user_message,
            "customer_name": customer_name,
            "satisfaction": None
        }
        df = pd.read_csv(CSV_FILE)
        df = pd.concat([df, pd.DataFrame([new_ticket])], ignore_index=True)
        df.to_csv(CSV_FILE, index=False)
        return classification, f"We apologize for the inconvenience. Might someone from support team can help you on this part for you reference a new ticket #{ticket_id} has been generated, and our team will follow up shortly.", ticket_id
    else:
        return classification, "Feedback handler received an unsupported type. Unable to understand your concern", None


### Query handler function

In [18]:
def handle_query(user_message):
    match = re.search(r'\b\d{6}\b', user_message)
    if match:
        ticket_id = int(match.group())
        df = pd.read_csv(CSV_FILE)
        ticket = df[df['ticket_id'] == ticket_id]
        if not ticket.empty:
            status = ticket.iloc[0]['status']
            return "Query", f"Your ticket #{ticket_id} is currently marked as: {status}.", ticket_id
        else:
            return "Query", f"Ticket #{ticket_id} was not found in our records. Please check the number or contact support.", None
    # Generic guidance for non-ticket queries
    if re.search(r'\b(status|balance|transfer|payment|card|block|fraud|loan|interest)\b', user_message.lower()):
        return "Query", "I can help with account-related queries. If you have a ticket number, include the 6-digit ID to check status. Otherwise, please provide more details.", None
    return "Query", "No valid ticket number found in your query. If you want to open a ticket, describe the issue and we'll create one.", None


### Ticket logger

In [19]:
def log_satisfaction(ticket_id, satisfaction):
    if ticket_id is None or pd.isna(ticket_id):
        return "No ticket associated with this interaction."
    df = pd.read_csv(CSV_FILE)
    if int(ticket_id) not in df['ticket_id'].astype(int).tolist():
        return f"Ticket #{int(ticket_id)} not found."
    df.loc[df['ticket_id'] == int(ticket_id), 'satisfaction'] = int(satisfaction)
    df.to_csv(CSV_FILE, index=False)
    return f"Satisfaction rating {int(satisfaction)}/5 logged for ticket #{int(ticket_id)}."

def view_tickets():
    df = pd.read_csv(CSV_FILE)
    if df.empty:
        return "No tickets in the system yet."
    return df.to_html(index=False)

### Initializing the Agents

In [21]:
# -------------------------
# LangGraph Agent State and Nodes
# -------------------------
class AgentState(TypedDict):
    input: str
    customer_name: Optional[str]
    classification: Optional[str]
    tool_name: Optional[str]
    tool_input: Optional[str]
    tool_output: Optional[str]
    answer: Optional[str]
    ticket_id: Optional[int]

def reason(state: AgentState) -> Dict[str, Any]:
    user_text = state.get("input", "")
    customer_name = state.get("customer_name", "Customer")
    # classification = ml_classify_message(user_text)
    classification = hf_model_classify(user_text)
    state["classification"] = classification

    if classification in ["Positive Feedback", "Negative Feedback"]:
        return {"tool_name": "feedback", "tool_input": {"message": user_text, "customer_name": customer_name}}
    elif classification == "Query":
        return {"tool_name": "query", "tool_input": {"message": user_text}}
    else:
        return {"answer": {"classification": classification, "response": "Unable to classify your request.", "ticket_id": None}}

def tool_executor(state: AgentState) -> Dict[str, Any]:
    tool_name = state.get("tool_name")
    tool_input = state.get("tool_input")
    if tool_name == "feedback":
        msg = tool_input["message"]
        cname = tool_input.get("customer_name", "Customer")
        classification, response, ticket_id = handle_feedback(msg, cname)
        return {"tool_output": {"classification": classification, "response": response, "ticket_id": ticket_id}}
    elif tool_name == "query":
        msg = tool_input["message"]
        classification, response, ticket_id = handle_query(msg)
        return {"tool_output": {"classification": classification, "response": response, "ticket_id": ticket_id}}
    else:
        return {"tool_output": {"classification": None, "response": "Unknown tool", "ticket_id": None}}

def finalize(state: AgentState) -> Dict[str, Any]:
    out = state.get("tool_output", {})
    classification = out.get("classification")
    response = out.get("response")
    ticket_id = out.get("ticket_id")
    return {"answer": {"classification": classification, "response": response, "ticket_id": ticket_id}}

### Graph: Nodes & Edges

In [23]:
# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("reason", reason)
workflow.add_node("tool_executor", tool_executor)
workflow.add_node("finalize", finalize)
workflow.set_entry_point("reason")

workflow.add_conditional_edges(
    "reason",
    lambda state: "tool" if state.get("tool_name") else "final",
    {"tool": "tool_executor", "final": END}
)
workflow.add_edge("tool_executor", "finalize")
workflow.add_edge("finalize", END)

app = workflow.compile()

In [28]:
state = {"input": 'How do I activate my debit card', "customer_name": 'Mayank'}
app.invoke(state)

{'input': 'How do I activate my debit card',
 'customer_name': 'Mayank',
 'tool_name': 'query',
 'tool_input': {'message': 'How do I activate my debit card'},
 'tool_output': {'classification': 'Query',
  'response': 'I can help with account-related queries. If you have a ticket number, include the 6-digit ID to check status. Otherwise, please provide more details.',
  'ticket_id': None},
 'answer': {'classification': 'Query',
  'response': 'I can help with account-related queries. If you have a ticket number, include the 6-digit ID to check status. Otherwise, please provide more details.',
  'ticket_id': None}}

### Gradio UI (shareable URL & Colab friendly)

In [29]:
# -------------------------
# Gradio UI (Colab friendly)
# -------------------------
with gr.Blocks() as demo:
    gr.Markdown("## 🏦 Banking Customer Support AI Agent")

    with gr.Row():
        user_message = gr.Textbox(label='Query', lines=3, placeholder="Enter your banking support message (or include a 6-digit ticket ID)...")
        user_name = gr.Textbox(label='Name', placeholder="Enter your name", value="Customer")

    with gr.Row():
        submit_btn = gr.Button("Submit")
        view_btn = gr.Button("View Tickets")

    # Initially hide classification and response until Submit is pressed
    classify_out = gr.Textbox(label="Classification", visible=False)
    response_out = gr.Textbox(label="Agent Response", visible=False)
    hidden_ticket_id = gr.Number(visible=False)

    # Satisfaction controls are hidden by default and only shown when a ticket exists
    satisfaction_slider = gr.Slider(minimum=1, maximum=5, step=1, label="Rate Satisfaction (1-5)", visible=False)
    # Buttons return the selected emoji string; map to score in callback
    # satisfaction = gr.Buttons(choices=["😞","😐","🙂","😊","😍"], label="How was the support?", visible=False)
    log_btn = gr.Button("Log Satisfaction", visible=False)
    # Log result hidden initially; will be shown only after the user clicks Log Satisfaction
    log_out = gr.Textbox(label="Satisfaction Logging Result", visible=False)

    tickets_html = gr.HTML(label="Stored Tickets")

    # Gradio callback that invokes the LangGraph app and controls visibility
    def invoke_agent(message, name):
        state = {"input": message, "customer_name": name}
        result = app.invoke(state)
        answer = result.get("answer", {}) or {}
        classification = answer.get("classification")
        response = answer.get("response")
        ticket_id = answer.get("ticket_id")

        # Prepare visibility updates:
        # Show classification & response after submit
        classify_update = gr.update(visible=True, value=classification or "")
        response_update = gr.update(visible=True, value=response or "")
        # Set hidden ticket id value (None if no ticket)
        ticket_update = gr.update(value=ticket_id if ticket_id is not None else None)
        # Show satisfaction controls only if a ticket was created/associated
        show_satisfaction = bool(ticket_id)
        slider_update = gr.update(visible=show_satisfaction, value=1 if show_satisfaction else None)
        logbtn_update = gr.update(visible=show_satisfaction)

        # Ensure the log result box is hidden when a new interaction starts (so it only appears after logging)
        log_out_update = gr.update(visible=False, value="")

        return classify_update, response_update, ticket_update, slider_update, logbtn_update, log_out_update

    # Wire submit to update classification/response and satisfaction visibility
    submit_btn.click(
        fn=invoke_agent,
        inputs=[user_message, user_name],
        outputs=[classify_out, response_out, hidden_ticket_id, satisfaction_slider, log_btn, log_out]
    )

    view_btn.click(
        fn=view_tickets,
        inputs=[],
        outputs=[tickets_html]
    )

    # When the user clicks Log Satisfaction, call log_satisfaction and show the result textbox only then
    def submit_satisfaction(ticket_id, rating):
        # Call the logging function and return a visible update for the log_out textbox
        msg = log_satisfaction(ticket_id, rating)
        return gr.update(visible=True, value=msg)

    log_btn.click(
        fn=submit_satisfaction,
        inputs=[hidden_ticket_id, satisfaction_slider],
        outputs=[log_out]
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://debcdc81c95492fb61.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_11585/259261008.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_ticket])], ignore_index=True)
/tmp/ipykernel_11585/259261008.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_ticket])], ignore_index=True)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e168507768417a0ecf.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://de5d4cfd7184b4e32e.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://635e5ba06abb3f24c9.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://debcdc81c95492fb61.gradio.live
